In [1]:
import polars as pl
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "scripts" / "RQ4" / "RQ4_plot_lib.py").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "scripts" / "RQ4"))

import importlib
import RQ4_plot_lib
importlib.reload(RQ4_plot_lib)  # pick up changes in the lib without restarting the kernel

from RQ4_plot_lib import (plot_brand_vs_free_prices, plot_price_difference, plot_brand_comparison, plot_avg_premium_per_brand,
                          plot_avg_premium_per_brand_periods, plot_daily_gap_2026, period_means_from_monthly, save_png)

data_dir = Path("/home/marvin/tankerkoenig-data/rq_brand_vs_free")

# 2014-2025: values of the first version of this notebook (the price files of those years are not available locally)
csv_path_brandvsfree = data_dir / "brand_vs_free_yearly_stats_2014_2025_from_notebook.csv"
csv_per_brand = data_dir / "per_brand_yearly_e10_ARAL_ESSO_OIL_2014_2025_from_notebook.csv"
csv_avg_premium = data_dir / "avg_premium_e10_per_brand_2014_2025_from_notebook.csv"

# 2026: computed with scripts/RQ4/RQ4_brands_vs_free_csvs.py (compute_2026_stats, same per-update means) per month;
# the two periods (before / after the 12 PM rule) are the mean of the monthly means
csv_2026_month = data_dir / "brand_vs_free_2026_monthly_stats.csv"
csv_2026_brand_month = data_dir / "per_brand_2026_monthly_stats.csv"
csv_2026_daily_gap = data_dir / "brand_vs_free_2026_daily_snapshot_gap.csv"  # compute_2026_snapshot_gap

In [2]:
# 2026 is split into two points: Jan-Mar (before the 12 PM rule) at x = 2026.0 and Apr-Oct (rule) at x = 2026.5,
# the red line for the rule (2026-04-01) sits in between.
RULE_X = 2026.25
period_x = {"2026 Jan-Mar": 2026.0, "2026 Apr-Oct": 2026.5}
fuels = ["diesel_mean", "e5_mean", "e10_mean"]

old_type = pl.read_csv(csv_path_brandvsfree).with_columns(pl.col("year").cast(pl.Float64))
new_type = (
    period_means_from_monthly(pl.read_csv(csv_2026_month), "station_type")
    .with_columns(pl.col("period").replace_strict(period_x).alias("year"))
    .select(["station_type", "year", *fuels])
)
df_brandvsfree = pl.concat([old_type, new_type]).sort(["station_type", "year"])

old_brand = pl.read_csv(csv_per_brand).with_columns(pl.col("year").cast(pl.Float64))
new_brand = (
    period_means_from_monthly(pl.read_csv(csv_2026_brand_month), "brand_normalized")
    .filter(pl.col("brand_normalized").is_in(["ARAL", "ESSO", "OIL!"]))
    .with_columns(pl.col("period").replace_strict(period_x).alias("year"))
    .select(["brand_normalized", "year", "e10_mean"])
)
df_per_brand = pl.concat([old_brand, new_brand]).sort(["brand_normalized", "year"])

brand = df_brandvsfree.filter(pl.col("station_type") == "brand_station").sort("year")
free  = df_brandvsfree.filter(pl.col("station_type") == "free_station").sort("year")

X_LABELS = [str(int(x)) if x < 2026 else ("2026<br>Jan-Mar" if x == 2026.0 else "2026<br>Apr-Oct") for x in brand["year"]]
print(df_brandvsfree.filter(pl.col("year") >= 2025))

shape: (6, 5)
┌───────────────┬────────┬─────────────┬──────────┬──────────┐
│ station_type  ┆ year   ┆ diesel_mean ┆ e5_mean  ┆ e10_mean │
│ ---           ┆ ---    ┆ ---         ┆ ---      ┆ ---      │
│ str           ┆ f64    ┆ f64         ┆ f64      ┆ f64      │
╞═══════════════╪════════╪═════════════╪══════════╪══════════╡
│ brand_station ┆ 2025.0 ┆ 1.6049      ┆ 1.7366   ┆ 1.6779   │
│ brand_station ┆ 2026.0 ┆ 1.858217    ┆ 1.89154  ┆ 1.832478 │
│ brand_station ┆ 2026.5 ┆ 2.178077    ┆ 2.168212 ┆ 2.109211 │
│ free_station  ┆ 2025.0 ┆ 1.5882      ┆ 1.7216   ┆ 1.6637   │
│ free_station  ┆ 2026.0 ┆ 1.840947    ┆ 1.877726 ┆ 1.820317 │
│ free_station  ┆ 2026.5 ┆ 2.163872    ┆ 2.157238 ┆ 2.100319 │
└───────────────┴────────┴─────────────┴──────────┴──────────┘


## Mean Fuel Prices: Brand vs. Free Stations (per year)

In [3]:
plot_brand_vs_free_prices(brand, free)

<Figure>

Brand stations charge a **consistent** premium of ~2 ct/L across all fuel types and all years. The gap is stable even during the 2022 energy price spike, suggesting it is structural rather than market-driven.

**2026:** the price level jumps with the Middle East war (Diesel from 1.60 € in 2025 to 1.86 € in Jan-Mar and 2.18 € in Apr-Oct), but brand and free stations move in parallel again. The gap is visible only as a small offset, see the next plot.

## Price Difference: Brand minus Free (per year)

In [4]:
plot_price_difference(brand, free, rule_x=RULE_X, x_labels=X_LABELS)

Diesel: avg difference 2014-2025 = +1.753 ct/L | 2026 Jan-Mar: +1.727, 2026 Apr-Oct: +1.420
E5: avg difference 2014-2025 = +1.621 ct/L | 2026 Jan-Mar: +1.381, 2026 Apr-Oct: +1.097
E10: avg difference 2014-2025 = +1.524 ct/L | 2026 Jan-Mar: +1.216, 2026 Apr-Oct: +0.889


<Figure>

The premium was between +1.2 and +1.9 ct/L in 2014-2025. Before the 12 PM rule, 2026 is still in that range (Jan-Mar: Diesel +1.7, E5 +1.4, E10 +1.2 ct/L). **After the rule the premium drops** (Apr-Oct: Diesel +1.4, E5 +1.1, E10 +0.9 ct/L), to the lowest values in the whole period for all three fuels. Brand stations still charge more, but compared with the 2014-2025 average the gap is ~20% smaller for Diesel, ~30% for E5 and ~40% for E10.

*2026 values are the mean of the monthly means (see the note in the results).*

## Price Difference: ARAL - other

In [5]:
plot_brand_comparison(df_per_brand, df_brandvsfree, fuel="e10_mean", brands=["ARAL", "OIL!", "ESSO"], rule_x=RULE_X, x_labels=X_LABELS)

<Figure>

**ARAL** is the most expensive brand in nearly **every year**, typically a few cents above both the brand average and free stations. **ESSO** briefly exceeded ARAL's prices in 2022–2023 during the energy crisis. **OIL!** stands out as an outlier — it consistently prices below even the average free station.

**2026:** ARAL stays the most expensive of the three (+2.7 / +2.6 ct/L above free stations before / after the 12 PM rule), ESSO's premium shrinks (+1.3 → +0.9). OIL! moves up from below the free stations (−0.3) to exactly their level (0.0) after the rule.

## Average Price Premium per Brand vs. Free Stations

In [6]:
avg_premium_old = pl.read_csv(csv_avg_premium)
free_monthly_2026 = pl.read_csv(csv_2026_month).filter(pl.col("station_type") == "free_station")

fig, premium_2026 = plot_avg_premium_per_brand_periods(avg_premium_old, pl.read_csv(csv_2026_brand_month), free_monthly_2026, fuel="e10_mean")
fig.show()
premium_2026.pivot(on="period", index="brand_normalized", values="difference_ct").sort("2026 Apr-Oct", descending=True)

<Figure>

brand_normalized,2026 Jan-Mar,2026 Apr-Oct
str,f64,f64
"""WESTFALEN""",2.429358,3.369157
"""ARAL""",2.72217,2.626632
"""Q1""",1.14859,2.128549
"""SHELL""",2.072793,1.467181
"""TOTAL""",1.151968,1.026472
…,…,…
"""HOYER""",1.220746,-0.719378
"""JET""",-0.684504,-0.732837
"""ENI""",-1.74868,-1.323669


**ARAL** charges the highest average premium at around +3 ct/L above free stations — the largest gap among all brands. Most other brands cluster between 0 and +2 ct/L. Notably, several brands including OIL! show a **negative** premium, meaning they are on average cheaper than independent free stations.

**2026 (grey: before, red: after the 12 PM rule):** 11 of 17 brands get relatively cheaper under the rule. The overall gap shrinks mainly because large networks lower their premium: SHELL (+2.1 → +1.5 ct/L), ESSO (+1.3 → +0.9), AGIP (+1.1 → +0.2), and ARAL slightly (+2.7 → +2.6), which stays among the most expensive brands. Exceptions are WESTFALEN (+2.4 → +3.4) and Q1 (+1.1 → +2.1), which become more expensive. HOYER (+1.2 → −0.7) and OMV (0.0 → −1.5) change a lot, but these are small networks and the values are noisy.

For 2026 the premium is compared within each month and then averaged. Pooling all updates of Jan-Mar would overweight March, when prices jumped by ~28 ct after the war began; HOYER for example reported 40% of its Jan-Mar updates in March (free stations: 33%) and would have appeared at +3.2 instead of +1.2 ct/L.

## Brand premium per day in 2026: 12 PM rule and fuel tax cuts

The plots above use the mean over all price updates, as in the original analysis (needed to compare with 2014-2025). For the question *"how much more do I pay at a brand station than at a free station"* this is not ideal: stations that change their price often count more, and since the 12 PM rule most updates happen at noon.

This plot therefore uses a **snapshot comparison**: every 15 minutes the price that is currently valid at each station is taken, brand and free stations are averaged and compared **at the same moment**, and the differences are averaged per day. Every station counts equally, and both groups always face the same market situation (noon jump, war, tax cut). Red line: 12 PM rule, shaded: fuel tax cuts.

In [7]:
daily_gap = pl.read_csv(csv_2026_daily_gap, try_parse_dates=True)
fig = plot_daily_gap_2026(daily_gap)
fig.show()

periods = {
    "Jan-Mar (before rule)": ("2026-01-01", "2026-03-31"),
    "Apr (rule, no tax cut)": ("2026-04-01", "2026-04-30"),
    "May-Jun (rule + tax cut 1)": ("2026-05-01", "2026-06-30"),
    "Jul-Sep (rule, no tax cut)": ("2026-07-01", "2026-09-30"),
    "Oct (rule + tax cut 2)": ("2026-10-01", "2026-10-31"),
}
pl.concat([
    daily_gap
    .filter(pl.col("day").is_between(pl.lit(a).str.to_date(), pl.lit(b).str.to_date()))
    .select([pl.lit(name).alias("period"), pl.col("diesel_gap_ct").mean(), pl.col("e5_gap_ct").mean(), pl.col("e10_gap_ct").mean()])
    for name, (a, b) in periods.items()
]).with_columns(pl.col("^.*_gap_ct$").round(2))

<Figure>

period,diesel_gap_ct,e5_gap_ct,e10_gap_ct
str,f64,f64,f64
"""Jan-Mar (before rule)""",3.32,2.9,2.58
"""Apr (rule, no tax cut)""",2.46,2.15,1.79
"""May-Jun (rule + tax cut 1)""",2.0,1.89,1.61
"""Jul-Sep (rule, no tax cut)""",2.34,2.07,1.68
"""Oct (rule + tax cut 2)""",1.62,1.37,1.05


- **12 PM rule:** before the rule the brand premium is ~3.3 ct/L for Diesel and ~2.6 ct/L for E10. In the first week after April 1 it even rises briefly (Diesel up to 4.1 ct/L), then from about April 8 it drops to a lower level and stays there: Apr-Oct on average ~2.2 ct/L for Diesel and ~1.6 ct/L for E10, about **1 ct/L less** than before.
- **Fuel tax cuts:** on the first day of both cuts (May 1, Oct 1) the difference turns **negative**: brand stations passed the cut on faster, free stations followed one day later. At the end of cut 1 (June 30 / July 1) it jumps to +3 ct/L: brand stations raised their prices faster, partly already at noon on June 30. Apart from these days the premium is only slightly smaller in the first two weeks of each cut (~0.3 ct/L) and then returns to its level. The tax cuts do not change the brand premium in a lasting way; October is the lowest month, but the gap was already falling in the second half of September.
- **Level:** in this comparison the premium is about twice as large as in the per-update yearly plots (E10 Jan-Mar: 2.6 vs. 1.2 ct/L). The per-update mean weights every station by how often it reports, and brand stations report more often (Jan-Mar: ~2,600 vs. ~1,800 updates per station), so the ~2 ct/L of the original analysis is rather a lower bound. The direction of all findings is the same with both methods.


## **Results**

This analysis compared fuel prices between **brand gas stations** (e.g. ARAL, ESSO, WESTFALEN) and **free/independent gas stations** in Germany from 2014 to October 2026.

**Brand stations are consistently more expensive** than free stations across all fuel types and all years, with an average difference in 2014-2025 of:

- **Diesel:** +1.75 ct/L
- **E5:** +1.62 ct/L
- **E10:** +1.53 ct/L

The price gap remained remarkably stable over time at roughly **~2 ct/L**, even during the energy price spike in 2022.

Looking at individual brands, **ARAL is the most expensive brand** with an average premium of around **+3 ct/L** compared to free stations. **ESSO** briefly surpassed ARAL as the most expensive brand in 2022 and 2023. On the other end of the spectrum, **OIL!** is consistently the cheapest brand — even undercutting the average free station price throughout the entire observation period.

**2026 and the 12 PM rule:** the premium persists, but it shrinks after the rule came into force on 2026-04-01 and reaches the lowest values of the whole period:

| | Diesel | E5 | E10 |
|---|---|---|---|
| 2014-2025 (avg) | +1.75 | +1.62 | +1.52 |
| 2026 Jan-Mar | +1.73 | +1.38 | +1.22 |
| 2026 Apr-Oct (12 PM rule) | +1.42 | +1.10 | +0.89 |

The daily snapshot comparison (every station counts equally, brand and free compared at the same moment) confirms the break: the premium drops by ~1 ct/L about one week after the rule came into force (E10: ~2.6 → ~1.6 ct/L, Diesel: ~3.3 → ~2.2 ct/L). In this comparison the premium is about twice as large as in the per-update yearly values, so the ~2 ct/L of the original analysis is a conservative estimate.

**Fuel tax cuts:** brand stations passed both cuts on faster (one day with a negative premium on May 1 and Oct 1) and also raised prices faster when cut 1 ended. Beyond these single days the cuts do not change the brand premium in a lasting way.

Overall, the results suggest that brand affiliation has a **small but persistent effect** on fuel prices, and that the brand premium is largely independent of broader market fluctuations. Regulation of *when* prices may rise, however, does affect it: since the 12 PM rule brand and free prices move closer together. The shrinking gap comes mainly from large networks like SHELL, ESSO and AGIP lowering their premium; ARAL only falls slightly (+2.7 → +2.6 ct/L), WESTFALEN even rises. A possible explanation is that all stations now raise prices at the same moment (12:00), so free stations can no longer profit from following the brand stations' increases with a delay.

*Data note:* 2014-2025 are the stored values of the first version (the yearly CSVs were not available). 2026 is computed with the same per-update means and the same station classification, but the two periods are the mean of the monthly means: pooling a period with strongly changing prices overweights months in which a group reports more often. The yearly values of 2014-2025 are pooled over the year and may be slightly affected by the same effect in volatile years like 2022; without the raw data they cannot be recomputed. Station classification: brands from the stations.parquet of March 2026, stations opened since then from the new stations.csv.
